In [ ]:
import numpy as np

def get_sgp_mat(num_in, num_out, link):
    A = np.zeros((num_in, num_out))
    for i, j in link:
        A[i, j] = 1
    A_norm = A / np.sum(A, axis=0, keepdims=True)
    return A_norm

# 返回的是节点自身的0阶邻居矩阵
def edge2mat(link, num_node):
    A = np.zeros((num_node, num_node))
    for i, j in link:
        A[j, i] = 1
    return A

def get_k_scale_graph(scale, A):
    if scale == 1:
        return A
    An = np.zeros_like(A)
    A_power = np.eye(A.shape[0])
    for k in range(scale):
        A_power = A_power @ A
        An += A_power
    An[An > 0] = 1
    return An

# 邻居矩阵乘以度矩阵的倒数
def normalize_digraph(A):
    Dl = np.sum(A, 0)
    h, w = A.shape
    Dn = np.zeros((w, w))
    for i in range(w):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** (-1)
    AD = np.dot(A, Dn)
    return AD

# inward方向和outward方向都需要做乘以度矩阵
# inward和outward
def get_spatial_graph(num_node, self_link, inward, outward):
    I = edge2mat(self_link, num_node) # 0 阶邻居矩阵
    In = normalize_digraph(edge2mat(inward, num_node)) # In邻接矩阵乘以度矩阵的倒数进行规范化
    Out = normalize_digraph(edge2mat(outward, num_node))
    A = np.stack((I, In, Out)) # A.shape = (3,25,25)
    return A

def normalize_adjacency_matrix(A):
    node_degrees = A.sum(-1)
    degs_inv_sqrt = np.power(node_degrees, -0.5)
    norm_degs_matrix = np.eye(len(node_degrees)) * degs_inv_sqrt
    return (norm_degs_matrix @ A @ norm_degs_matrix).astype(np.float32)


def k_adjacency(A, k, with_self=False, self_factor=1):
    assert isinstance(A, np.ndarray)
    I = np.eye(len(A), dtype=A.dtype)
    if k == 0:
        return I
    Ak = np.minimum(np.linalg.matrix_power(A + I, k), 1) \
       - np.minimum(np.linalg.matrix_power(A + I, k - 1), 1)
    if with_self:
        Ak += (self_factor * I)
    return Ak

def get_multiscale_spatial_graph(num_node, self_link, inward, outward):
    I = edge2mat(self_link, num_node)
    A1 = edge2mat(inward, num_node)
    A2 = edge2mat(outward, num_node)
    A3 = k_adjacency(A1, 2)
    A4 = k_adjacency(A2, 2)
    A1 = normalize_digraph(A1)
    A2 = normalize_digraph(A2)
    A3 = normalize_digraph(A3)
    A4 = normalize_digraph(A4)
    A = np.stack((I, A1, A2, A3, A4))
    return A


num_node = 25

self_link = [(i, i) for i in range(num_node)]

inward = [
    (1, 0), (2, 1), (3, 2), (4, 3),
    (5, 1), (6, 5), (7, 6),
    (8, 1), (9, 8), (10, 9),
    (11, 8), (12, 11), (13, 12),
    (14, 0), (15, 0),
    (16, 14), (17, 15),
    (18, 14), (19, 18), (20, 19),
    (21, 14), (22, 11),
    (23, 22), (24, 11)
]

outward = [(j, i) for (i, j) in inward]

neighbor = inward + outward


def edge2mat(link, num_node):
    A = np.zeros((num_node, num_node))
    for i, j in link:
        A[j, i] = 1
    return A


def normalize_digraph(A):
    Dl = np.sum(A, 0)
    num_node = A.shape[0]
    Dn = np.zeros((num_node, num_node))
    for i in range(num_node):
        if Dl[i] > 0:
            Dn[i, i] = Dl[i] ** (-1)
    AD = np.dot(A, Dn)
    return AD


def get_spatial_graph(num_node, self_link, inward, outward):

    I = edge2mat(self_link, num_node)

    In = normalize_digraph(edge2mat(inward, num_node))

    Out = normalize_digraph(edge2mat(outward, num_node))

    A = np.stack((I, In, Out))

    return A


class Graph:

    def __init__(self, labeling_mode='spatial'):

        self.num_node = num_node
        self.self_link = self_link
        self.inward = inward
        self.outward = outward
        self.neighbor = neighbor

        self.A = self.get_adjacency_matrix(labeling_mode)

    def get_adjacency_matrix(self, labeling_mode=None):

        if labeling_mode is None:
            return self.A

        if labeling_mode == 'spatial':
            A = get_spatial_graph(
                self.num_node,
                self.self_link,
                self.inward,
                self.outward
            )

        else:
            raise ValueError("Unsupported labeling mode")

        return A

In [ ]:
import math
import pdb

import numpy as np
import torch
import torch.nn as nn
from torch.autograd import Variable

def import_class(name):
    components = name.split('.')
    mod = __import__(components[0])
    for comp in components[1:]:
        mod = getattr(mod, comp)
    return mod


def conv_branch_init(conv, branches):
    weight = conv.weight
    # weight.size ->  (out_channels, input_channels/groups, kernel_size[0], kernel_size[1])
    n = weight.size(0)
    k1 = weight.size(1)
    k2 = weight.size(2)
    nn.init.normal_(weight, 0, math.sqrt(2. / (n * k1 * k2 * branches)))
    nn.init.constant_(conv.bias, 0)


def conv_init(conv):
    if conv.weight is not None:
        nn.init.kaiming_normal_(conv.weight, mode='fan_out')
        # 使用kaiming正态分布初始化卷积层参数，通过创建随机矩阵显式创建权重，则应进行设置mode=‘fan_out’
    if conv.bias is not None:
        nn.init.constant_(conv.bias, 0)


def bn_init(bn, scale):
    nn.init.constant_(bn.weight, scale)
    nn.init.constant_(bn.bias, 0)


def weights_init(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        if hasattr(m, 'weight'):
            nn.init.kaiming_normal_(m.weight, mode='fan_out')
        if hasattr(m, 'bias') and m.bias is not None and isinstance(m.bias, torch.Tensor):
            nn.init.constant_(m.bias, 0)
    elif classname.find('BatchNorm') != -1:
        if hasattr(m, 'weight') and m.weight is not None:
            m.weight.data.normal_(1.0, 0.02)
        if hasattr(m, 'bias') and m.bias is not None:
            m.bias.data.fill_(0)


class DynamicFrameWeighting(nn.Module):
    def __init__(self, in_channels):
        super(DynamicFrameWeighting, self).__init__()
        self.conv_ta = nn.Conv2d(in_channels, 1, kernel_size=(5, 1), padding=(2, 0))
        nn.init.constant_(self.conv_ta.weight, 0)
        nn.init.constant_(self.conv_ta.bias, 0)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_out = torch.mean(x, dim=-1, keepdim=True)
        # max_out, _ = torch.max(x, dim=1, keepdim=True)
        # x = torch.cat([avg_out, max_out], dim=1)
        # N, C, T, V -> N, C, T, 1
        att = self.sigmoid(self.conv_ta(avg_out))   # N, 1, T, 1
        x = x * att + x
        return x


class TemporalConv(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, dilation=1):
        super(TemporalConv, self).__init__()
        pad = (kernel_size + (kernel_size-1) * (dilation-1) - 1) // 2
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(kernel_size, 1),
            padding=(pad, 0),
            stride=(stride, 1),
            dilation=(dilation, 1))

        self.bn = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        return x


class J_GTM(nn.Module):
    def __init__(self, in_channels, out_channels, stride, num_heads):
        super(J_GTM, self).__init__()

        # Temporal Self-Attention Module
        self.dim_k = out_channels 
        self.dim_v = out_channels
        self.stride = stride
        self.num_heads = num_heads
        self.conv_q = nn.Conv2d(in_channels, out_channels, kernel_size=(1, 1))
        self.conv_k = nn.Conv2d(in_channels, out_channels, kernel_size=(1, 1))
        self.conv_v = nn.Conv2d(in_channels, out_channels, kernel_size=(1, 1))
        self.conv_1 = nn.Conv2d(in_channels, out_channels, kernel_size=(3, 1), stride=(stride, 1), padding=(1, 0))
        self.tanh = nn.Tanh()
        self.bn1 = nn.BatchNorm2d(out_channels)

        conv_init(self.conv_q)
        conv_init(self.conv_k)
        conv_init(self.conv_v)
        conv_init(self.conv_1)
        bn_init(self.bn1, 1)


    def forward(self, x):
        # TSA分支
        x_tsa = x
        N, C, T, V = x_tsa.shape
        nh = self.num_heads
        dk = self.dim_k // nh
        dv = self.dim_v // nh
        q = (self.conv_q(x_tsa)).permute(0, 3, 2, 1).contiguous()  # N*M, C, T, V ->  N*M, V, T, C
        q = q.reshape(N, V, T, nh, dk).transpose(2, 3)  # N, V, nh, T, dk
        q = q.reshape(N, V*nh, T, dk)
        k = (self.conv_k(x_tsa)).permute(0, 3, 2, 1).contiguous()  # N*M, C, T, V ->  N*M, V, T, C
        k = k.reshape(N, V, T, nh, dk).transpose(2, 3)  # N, V, nh, T, dk
        k = k.reshape(N, V*nh, T, dk)
        v = (self.conv_v(x_tsa)).permute(0, 3, 2, 1).contiguous()  # N*M, C, T, V ->  N*M, V, T, C
        v = v.reshape(N, V, T, nh, dv).transpose(2, 3)  # N, V, nh, T, dk
        v = v.reshape(N, V*nh, T, dv)

        dist = torch.matmul(q, k.transpose(2, 3)) / math.sqrt(dk)
        dist = self.tanh(dist)   # N, V*nh, T, T
        att = torch.matmul(dist, v)   # N, V*nh, T, dv
        att = att.reshape(N, V, nh, T, dv).transpose(2, 3).reshape(N, V, T, self.dim_v)
        out_tsa = att.permute(0, 3, 2, 1).contiguous()
        
        out_tsa = self.bn1(self.conv_1(out_tsa))  # N*M, C, T/2, V
        return out_tsa


class LG_TCN_unit(nn.Module):
    def __init__(self,
                 in_channels,
                 out_channels,
                 kernel_size=3,
                 stride=1,
                 dilations=[1, 2, 3, 4],
                 residual=True,
                 residual_kernel_size=1,
                 num_heads=4):

        super().__init__()
        assert out_channels % (len(dilations) + 2) == 0, '# out channels should be multiples of # branches'

        self.temporal_weight = DynamicFrameWeighting(in_channels)
        # Multiple branches of temporal convolution
        self.num_branches = len(dilations) + 2
        # 模型除了两条带有dilation的分支外，还有一个Maxpool 和 普通的1*1卷积，所以+2
        branch_channels = out_channels // self.num_branches
        # 计算每个分支的平均维度，这样所有分支结果进行concat之后维度与out_channels一致
        if type(kernel_size) == list:
            assert len(kernel_size) == len(dilations)
        else:
            kernel_size = [kernel_size]*len(dilations)

        # Temporal Convolution branches
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    branch_channels,
                    kernel_size=1,
                    padding=0),
                nn.BatchNorm2d(branch_channels),
                nn.ReLU(inplace=True),
                TemporalConv(
                    branch_channels,
                    branch_channels,
                    kernel_size=ks,
                    stride=stride,
                    dilation=dilation),
            )
            for ks, dilation in zip(kernel_size, dilations)
        ])
        
        # Additional Max & 1x1 branch
        self.branches.append(nn.Sequential(
            nn.Conv2d(in_channels, branch_channels, kernel_size=1, padding=0),
            nn.BatchNorm2d(branch_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(3, 1), stride=(stride, 1), padding=(1, 0)),
            nn.BatchNorm2d(branch_channels) 
        ))

        self.branches.append(nn.Sequential(
            nn.Conv2d(in_channels, branch_channels, kernel_size=1, padding=0, stride=(stride, 1)),
            nn.BatchNorm2d(branch_channels)
        ))
        
        self.branches.append(nn.Sequential(
                J_GTM(
                    branch_channels,
                    branch_channels,
                    stride=stride,
                    num_heads=num_heads)
        ))
        self.branches.append(nn.Sequential(
                J_GTM(
                    branch_channels,
                    branch_channels,
                    stride=stride,
                    num_heads=num_heads)
        ))  

        self.a = nn.Parameter(torch.zeros(1))
        self.b = nn.Parameter(torch.zeros(1))
        self.bn1 = nn.BatchNorm2d(branch_channels)
        self.bn2 = nn.BatchNorm2d(branch_channels)

        # Residual connection
        if not residual:
            self.residual = lambda x: 0
        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x
        else:
            self.residual = TemporalConv(in_channels, out_channels, kernel_size=residual_kernel_size, stride=stride)

        # initialize
        self.apply(weights_init)
        bn_init(self.bn1, 1)
        bn_init(self.bn2, 1)

        
    def forward(self, x):
        # Input dim: (N,C,T,V)
        res = self.residual(x)
        x = self.temporal_weight(x)
        branch_outs = []
        x1 = self.branches[0][0](x)
        x1 = self.branches[0][1](x1)
        x1 = self.branches[0][2](x1)
        x1_1 = self.branches[0][3](x1)
        x1_2 = self.branches[4](x1)
        x1 = self.bn1(x1_1 + x1_2 * self.a)
        x2 = self.branches[1][0](x)
        x2 = self.branches[1][1](x2)
        x2 = self.branches[1][2](x2)
        x2_1 = self.branches[1][3](x2)
        x2_2 = self.branches[5](x2)
        x2 = self.bn2(x2_1 + x2_2 * self.b)
        x3 = self.branches[2](x)
        x4 = self.branches[3](x)
        out = torch.cat((x1, x2, x3, x4), dim=1)
        # 横向concat连接
        out += res
        return out


class LG_GC(nn.Module):
    def __init__(self, in_channels, out_channels, rel_reduction=8, mid_reduction=1):
        super(LG_GC, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        if in_channels == 3 or in_channels == 9:
            self.rel_channels = 8
            self.mid_channels = 16
        else:
            self.rel_channels = 8
            self.mid_channels = 16
            # //表示 除 之后向下取整，不大于商的最大整数
        self.conv1 = nn.Conv2d(self.in_channels, self.rel_channels, kernel_size=1)
        self.conv2 = nn.Conv2d(self.in_channels, self.rel_channels, kernel_size=1)
        # 1*1卷积改变维度，相当于T*N*C -> T*N*C/r
        self.conv3 = nn.Conv2d(self.in_channels, self.out_channels, kernel_size=1)
        self.conv4 = nn.Conv2d(self.rel_channels, self.out_channels, kernel_size=1)
        self.conv5 = nn.Conv2d(self.in_channels, self.rel_channels, kernel_size=1)
        self.conv6 = nn.Conv2d(1, 1, kernel_size=1)
        self.conv7 = nn.Conv2d(1, 1, kernel_size=1)
        # self.conv6 = nn.Conv2d(self.in_channels, self.rel_channels, kernel_size=1)
        self.tanh = nn.Tanh()
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                conv_init(m)
                # isinstance用来判断m与后面一个参数的类型（也就是是卷积还是bn操作）是否相同，再考虑进行初始化
            elif isinstance(m, nn.BatchNorm2d):
                bn_init(m, 1)

    def forward(self, x, A=None, alpha=1, beta=1):
        x1, x2, x3 = self.conv1(x).mean(-2), self.conv2(x).mean(-2), self.conv3(x)
        x1 = self.tanh(x1.unsqueeze(-1) - x2.unsqueeze(-2))
        # 表示M()函数计算Xi与Xj之间的通道特定关系 x1: N*C*V*1 - N*C*1*V -> N*C/r*V*V
        x_attention_source = self.conv5(x)  # (N,rel_channels,T,V)
        N, C, T, V = x_attention_source.size()
        att_output_of_each_channels = []
        for i in range(x_attention_source.size()[1]):
            channel = x_attention_source[:, i]  # (N,T,V)
            channel = torch.mean(channel, 1, keepdim=True).unsqueeze(1)  # (N,T,V) -> (N,1,V) -> (N,1,1,V)
            channel = channel.transpose(2, 3)  # (N,1,V,1)
            channel = torch.repeat_interleave(channel, 25, 3)
            channel = channel.reshape(N, 1, V, 25)
            channel_q = self.conv6(channel).permute(0, 2, 3, 1).contiguous().flatten(2)  # (N,V,25,1) -> (N, V, 25)
            channel_k = self.conv7(channel).permute(0, 2, 3, 1).contiguous().flatten(2)  # (N,V,25,1) -> (N, V, 25)
            att = torch.matmul(channel_q, channel_k.transpose(-1, -2))  # N,V,V
            att_scores = att / math.sqrt(25.0)
            att_res = torch.softmax(att_scores, dim=-1)   # N, V, V
            att_output_of_each_channels.append(att_res)

        res = self.tanh(torch.stack(att_output_of_each_channels, 1))  # N C V V

        x1 = self.conv4(x1) * alpha + (A.unsqueeze(0).unsqueeze(0) if A is not None else 0) + (self.conv4(res) if res is not None else 0) * beta       # N,C,V,V
        # conv4的作用：M（）函数计算相关性之后，通过1*1卷积将Q维度改为C_out，从而能与输入特征X的新维度C_out对应
        x1 = torch.einsum('ncuv,nctv->nctu', x1, x3)
        # 爱因斯坦求和约定，对前 -> 后消除的字母进行求和，即对v进行求和
        return x1
        # 得到的Q矩阵与conv3卷积后输入特征相乘，得到输出特征
    

class unit_tcn(nn.Module):
    # 该普通时间卷积是为了给残差连接进行时间卷积
    def __init__(self, in_channels, out_channels, kernel_size=9, stride=1):
        super(unit_tcn, self).__init__()
        pad = int((kernel_size - 1) / 2)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=(kernel_size, 1), padding=(pad, 0),
                              stride=(stride, 1))

        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        conv_init(self.conv)
        bn_init(self.bn, 1)

    def forward(self, x):
        x = self.bn(self.conv(x))
        return x


class LG_GCN_unit(nn.Module):
    def __init__(self, in_channels, out_channels, A, coff_embedding=4, adaptive=True, residual=True):
        super(LG_GCN_unit, self).__init__()
        inter_channels = out_channels // coff_embedding
        self.inter_c = inter_channels
        self.out_c = out_channels
        self.in_c = in_channels
        self.adaptive = adaptive
        self.num_subset = A.shape[0]
        self.convs = nn.ModuleList()
        for i in range(self.num_subset):
            self.convs.append(LG_GC(in_channels, out_channels))

        if residual:
            if in_channels != out_channels:
                self.down = nn.Sequential(
                    nn.Conv2d(in_channels, out_channels, 1),
                    nn.BatchNorm2d(out_channels)
                )
            else:
                self.down = lambda x: x
        else:
            self.down = lambda x: 0
        if self.adaptive:
            self.PA = nn.Parameter(torch.from_numpy(A.astype(np.float32)))
            # 将PA添加到参数列表中，送入优化器随着训练一起学习更新；后面from_numpy等 是将数据类型修改为float32的数组A，改为张量类型
        else:
            self.A = Variable(torch.from_numpy(A.astype(np.float32)), requires_grad=False)
            # requires_grad是参不参与误差反向传播, 要不要计算梯度
        self.alpha = nn.Parameter(torch.zeros(1))
        self.beta = nn.Parameter(torch.zeros(1))
        self.bn = nn.BatchNorm2d(out_channels)
        self.soft = nn.Softmax(-2)
        self.relu = nn.ReLU(inplace=True)

        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                conv_init(m)
            elif isinstance(m, nn.BatchNorm2d):
                bn_init(m, 1)
        bn_init(self.bn, 1e-6)

    def forward(self, x):
        y = None
        if self.adaptive:
            A = self.PA
        else:
            A = self.A.cuda(x.get_device())
        for i in range(self.num_subset):
            z = self.convs[i](x, A[i], self.alpha, self.beta)
            y = z + y if y is not None else z
        y = self.bn(y)
        y += self.down(x)
        y = self.relu(y)
        return y


class LG_SGNet_Block(nn.Module):
    def __init__(self, in_channels, out_channels, A, stride=1, residual=True, adaptive=True, kernel_size=5, dilations=[1,2], num_heads=4):
        super(LG_SGNet_Block, self).__init__()
        self.gcn1 = LG_GCN_unit(in_channels, out_channels, A, adaptive=adaptive)
        self.tcn1 = LG_TCN_unit(out_channels, out_channels, kernel_size=kernel_size, stride=stride, dilations=dilations,
                                            residual=False, num_heads=num_heads)
        self.relu = nn.ReLU(inplace=True)
        if not residual:
            self.residual = lambda x: 0

        elif (in_channels == out_channels) and (stride == 1):
            self.residual = lambda x: x

        else:
            self.residual = unit_tcn(in_channels, out_channels, kernel_size=1, stride=stride)

    def forward(self, x):
        y = self.relu(self.tcn1(self.gcn1(x)) + self.residual(x))
        return y


class Model(nn.Module):
    def __init__(self, num_class=3, num_point=25, num_person=9, graph=None, graph_args=dict(), in_channels=2,
                 drop_out=0, adaptive=True):
        super(Model, self).__init__()

        if graph is None:
            raise ValueError()
        # 自定义的异常处理
        else:
            self.graph = Graph(**graph_args)

        A = self.graph.A # 3,25,25   一开始是 3  channels

        self.num_class = num_class
        self.num_point = num_point
        self.data_bn = nn.BatchNorm1d(num_person * in_channels * num_point)

        base_channel = 64
        self.l1 = LG_SGNet_Block(in_channels, base_channel, A, residual=False, adaptive=adaptive)
        self.l2 = LG_SGNet_Block(base_channel, base_channel, A, adaptive=adaptive)
        self.l3 = LG_SGNet_Block(base_channel, base_channel, A, adaptive=adaptive)
        self.l4 = LG_SGNet_Block(base_channel, base_channel, A, adaptive=adaptive)
        self.l5 = LG_SGNet_Block(base_channel, base_channel*2, A, stride=2, adaptive=adaptive)
        self.l6 = LG_SGNet_Block(base_channel*2, base_channel*2, A, adaptive=adaptive)
        self.l7 = LG_SGNet_Block(base_channel*2, base_channel*2, A, adaptive=adaptive)
        self.l8 = LG_SGNet_Block(base_channel*2, base_channel*4, A, stride=2, adaptive=adaptive)
        self.l9 = LG_SGNet_Block(base_channel*4, base_channel*4, A, adaptive=adaptive)
        self.l10 = LG_SGNet_Block(base_channel*4, base_channel*4, A, adaptive=adaptive)

        self.fc = nn.Linear(base_channel*4, num_class)
        nn.init.normal_(self.fc.weight, 0, math.sqrt(2. / num_class))
        bn_init(self.data_bn, 1)
        if drop_out:
            self.drop_out = nn.Dropout(drop_out)
        else:
            self.drop_out = lambda x: x

    def forward(self, x):
        if len(x.shape) == 3:
            N, T, VC = x.shape
            x = x.view(N, T, self.num_point, -1).permute(0, 3, 1, 2).contiguous().unsqueeze(-1)
            # 先将x的维度改为N,T,V,C,然后根据索引调换位置 -> N,C,T,V,在最后的索引位置上添加一个 M 维度
        N, C, T, V, M = x.size()

        x = x.permute(0, 4, 3, 1, 2).contiguous().view(N, M * V * C, T)
        x = self.data_bn(x)
        x = x.view(N, M, V, C, T).permute(0, 1, 3, 4, 2).contiguous().view(N * M, C, T, V)
        x = self.l1(x)
        x = self.l2(x)
        x = self.l3(x)
        x = self.l4(x)
        x = self.l5(x)
        x = self.l6(x)
        x = self.l7(x)
        x = self.l8(x)
        x = self.l9(x)
        x = self.l10(x)

        # N*M,C,T,V
        c_new = x.size(1)
        x = x.view(N, M, c_new, -1)
        x = x.mean(3).mean(1)
        x = self.drop_out(x)

        return self.fc(x)

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score

from tqdm import tqdm

def normalize_skeleton(X, width=1280, height=720):
    """
    X shape: (N, T, M, V, C)
    C = (x, y, confidence)
    """

    X = X.copy()

    # center theo hip (joint 8)
    hip = X[:, :, :, 8:9, :2]
    X[:, :, :, :, :2] = X[:, :, :, :, :2] - hip

    # scale theo frame size
    X[:, :, :, :, 0] /= width
    X[:, :, :, :, 1] /= height

    return X
# ==================================
# DATASET CLASS
# ==================================

class MyDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):

        x = torch.tensor(self.X[idx], dtype=torch.float32)
        y = torch.tensor(self.y[idx], dtype=torch.long)

        return x, y, idx


# ==================================
# METRICS (3 CLASS)
# ==================================

def print_metrics(y_true, y_pred):

    cm = confusion_matrix(y_true, y_pred)

    print("\nConfusion Matrix")
    print(cm)

    acc = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true,
        y_pred,
        average="macro"
    )

    recall = recall_score(
        y_true,
        y_pred,
        average="macro"
    )

    f1 = f1_score(
        y_true,
        y_pred,
        average="macro"
    )

    print("\nMetrics")

    print(f"Accuracy : {acc*100:.2f}%")
    print(f"Precision: {precision*100:.2f}%")
    print(f"Recall   : {recall*100:.2f}%")
    print(f"F1-score : {f1*100:.2f}%")


# ==================================
# PROCESSOR
# ==================================

class Processor:

    def __init__(self, model, arg):

        self.arg = arg

        self.model = model.cuda()

        self.output_device = arg.device if isinstance(arg.device, int) else arg.device[0]

        self.best_acc = 0
        self.best_acc_epoch = 0
        self.start_epoch = arg.start_epoch

        os.makedirs(arg.work_dir, exist_ok=True)

        # ==================================
        # LOAD DATASET
        # ==================================

        print("Loading dataset...")

        # =====================
        # TRAIN
        # =====================
        
        X_train = np.load(
            '../X_train.npy'
        )
        
        y_train = np.load(
            '../y_train.npy'
        )
        
        # =====================
        # VAL
        # =====================
        
        X_val = np.load(
            '../X_val.npy'
        )
        
        y_val = np.load(
            '../y_val.npy'
        )
        
        # =====================
        # TEST
        # =====================
        
        test_X = np.load(
            '../X_test.npy'
        )
        
        test_y = np.load(
            '../y_test.npy'
        )
        print("Normalizing skeleton data...")

        X_train = normalize_skeleton(X_train, 1280, 720)
        X_val   = normalize_skeleton(X_val, 1280, 720)
        test_X  = normalize_skeleton(test_X, 1280, 720)
        # =====================
        # TRANSPOSE
        # (N,T,M,V,C) → (N,C,T,V,M)
        # =====================
        
        X_train = np.transpose(X_train, (0,4,1,3,2))
        X_val   = np.transpose(X_val,   (0,4,1,3,2))
        test_X  = np.transpose(test_X,  (0,4,1,3,2))
        X_train = X_train[:, :2, :, :, :]
        X_val   = X_val[:, :2, :, :, :]
        test_X  = test_X[:, :2, :, :, :]
        print("Train:", X_train.shape)
        print("Val:", X_val.shape)
        print("Test:", test_X.shape)
        # ==================================
        # DATALOADER
        # ==================================

        self.data_loader = {

            'train': DataLoader(
                MyDataset(X_train, y_train),
                batch_size=arg.batch_size,
                shuffle=True,
                num_workers=2
            ),

            'val': DataLoader(
                MyDataset(X_val, y_val),
                batch_size=arg.test_batch_size,
                shuffle=False,
                num_workers=2
            ),

            'test': DataLoader(
                MyDataset(test_X, test_y),
                batch_size=arg.test_batch_size,
                shuffle=False,
                num_workers=2
            )
        }

        # ==================================
        # LOSS + OPTIMIZER
        # ==================================

        self.loss = nn.CrossEntropyLoss().cuda(self.output_device)

        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr=arg.base_lr,
            weight_decay=arg.weight_decay
        )

        # ==================================
        # RESUME TRAINING
        # ==================================

        resume_path = os.path.join(
            arg.work_dir,
            "latest_checkpoint.pt"
        )

        if os.path.exists(resume_path):

            checkpoint = torch.load(
                resume_path,
                weights_only=False
            )

            self.model.load_state_dict(
                checkpoint['model_state']
            )

            self.optimizer.load_state_dict(
                checkpoint['optim_state']
            )

            self.best_acc = checkpoint['best_acc']
            self.best_acc_epoch = checkpoint['best_acc_epoch']

            self.start_epoch = checkpoint['epoch'] + 1

            print(
                f"Resumed from epoch {self.start_epoch}"
            )

        else:

            print("Training from scratch")


    # ==================================
    # TRAIN
    # ==================================

    def train(self, epoch):

        self.model.train()

        print(f"\nEpoch {epoch+1} Training")

        loader = self.data_loader['train']

        loss_value = []
        acc_value = []

        for data, label, _ in tqdm(loader, ncols=60):

            data = data.cuda()
            label = label.cuda()

            output = self.model(data)

            loss = self.loss(output, label)

            self.optimizer.zero_grad()

            loss.backward()

            self.optimizer.step()

            _, pred = torch.max(output, 1)

            acc = torch.mean(
                (pred == label).float()
            )

            loss_value.append(loss.item())
            acc_value.append(acc.item())

        print(
            f"Train loss: {np.mean(loss_value):.4f}"
        )

        print(
            f"Train acc : {np.mean(acc_value)*100:.2f}%"
        )


    # ==================================
    # EVAL
    # ==================================

    def eval(self, epoch, mode='val'):

        self.model.eval()

        loader = self.data_loader[mode]

        loss_value = []
        score_frag = []
        label_list = []

        with torch.no_grad():

            for data, label, _ in tqdm(loader, ncols=60):

                data = data.cuda()
                label = label.cuda()

                output = self.model(data)

                loss = self.loss(output, label)

                loss_value.append(loss.item())

                score_frag.append(
                    output.cpu().numpy()
                )

                label_list.append(
                    label.cpu().numpy()
                )

        score = np.concatenate(score_frag)

        label_list = np.concatenate(label_list)

        acc = accuracy_score(
            label_list,
            np.argmax(score, axis=1)
        )

        print(
            f"{mode} loss: {np.mean(loss_value):.4f}"
        )

        print(
            f"{mode} acc : {acc*100:.2f}%"
        )

        if mode == 'val' and acc > self.best_acc:

            self.best_acc = acc
            self.best_acc_epoch = epoch + 1

            torch.save(
                self.model.state_dict(),
                os.path.join(
                    self.arg.work_dir,
                    'best_model.pt'
                )
            )

            print("New best model saved")

        return acc


    # ==================================
    # START TRAINING
    # ==================================

    def start(self):

        for epoch in range(
            self.start_epoch,
            self.arg.num_epoch
        ):

            self.train(epoch)

            self.eval(epoch, 'val')

            torch.save({

                'epoch': epoch,

                'model_state': self.model.state_dict(),

                'optim_state': self.optimizer.state_dict(),

                'best_acc': self.best_acc,

                'best_acc_epoch': self.best_acc_epoch

            },

            os.path.join(
                self.arg.work_dir,
                "latest_checkpoint.pt"
            ))

        print(
            f"\nBest val acc {self.best_acc*100:.2f}% "
            f"at epoch {self.best_acc_epoch}"
        )


    # ==================================
    # TEST BEST MODEL
    # ==================================

    def test_best(self):

        print("\nEvaluating BEST model")

        best_model_path = os.path.join(
            self.arg.work_dir,
            'best_model.pt'
        )

        self.model.load_state_dict(
            torch.load(best_model_path)
        )

        self.model.eval()

        all_labels = []
        all_preds = []

        with torch.no_grad():

            for data, label, _ in self.data_loader['test']:

                data = data.cuda()
                label = label.cuda()

                output = self.model(data)

                _, pred = torch.max(output, 1)

                all_labels.append(
                    label.cpu().numpy()
                )

                all_preds.append(
                    pred.cpu().numpy()
                )

        all_labels = np.concatenate(all_labels)
        all_preds = np.concatenate(all_preds)

        print_metrics(
            all_labels,
            all_preds
        )

In [ ]:
class Args:
    device = 0
    batch_size = 16
    test_batch_size = 32
    base_lr = 0.0001
    weight_decay = 1e-4
    num_epoch = 100
    start_epoch = 0
    save_interval = 15
    model_saved_name = 'my_model'
    work_dir = './results'

arg = Args()

model = Model(
    num_class=7,
    num_point=25,
    num_person=9,
    in_channels=2,
    graph=Graph,
    graph_args={'labeling_mode': 'spatial'}
)


In [ ]:
processor = Processor(model, arg)
processor.start()      
processor.test_best()  
